# 02 — Build the Corral transition table

Project the 2025 XLSX into the staging-table shape described in `03_transition_table_schema.sql`. One row per XLSX row. Carries the FK back to `dbo.TdrTransaction` plus the 8 XLSX columns Corral doesn't source.

Outputs:
- `out/corral_transition_table.csv`
- `out/corral_transition_table.parquet`
- `out/transition_table_summary.json` — row count + `LinkageStatus` distribution.

Read-only. No writes to Corral.

In [ ]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from sqlalchemy import text

NB_DIR = Path.cwd()
REPO_ROOT = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
OUT = NB_DIR / "out" if NB_DIR.name == "notebooks" else NB_DIR / "notebooks" / "out"
OUT.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO_ROOT / "erd"))
from db_corral import get_engine  # noqa: E402

XLSX = REPO_ROOT / "data" / "raw_data" / "2025 Transactions and Allocations Details.xlsx"
assert XLSX.exists(), f"XLSX not found: {XLSX}"
LOADED_AT = datetime.now(timezone.utc).isoformat()
print(f"XLSX: {XLSX.name}")
print(f"LoadedAt: {LOADED_AT}")

## Load

In [ ]:
xlsx = pd.read_excel(XLSX, dtype=str)
xlsx = xlsx.reset_index().rename(columns={"index": "SourceRowNumber"})
print(f"XLSX rows: {len(xlsx):,}")
xlsx.head(3)

## Canonical join key

Parse `TdrTransactionID` from the synthetic `TransactionID` (`{LeadAgency}-{TxnType}-{TdrTransactionID}`). Rows without a `TransactionID` get NULL — those are XLSX-only orphans.

In [ ]:
def parse_tdr_id(tid: object) -> int | None:
    if tid is None or (isinstance(tid, float) and pd.isna(tid)):
        return None
    s = str(tid).strip()
    if not s:
        return None
    parts = s.split("-")
    if len(parts) < 3:
        return None
    try:
        return int(parts[-1])
    except ValueError:
        return None

xlsx["ParsedTdrTransactionID"] = xlsx["TransactionID"].map(parse_tdr_id)
print("Parsed TdrTransactionID populated:", xlsx["ParsedTdrTransactionID"].notna().sum(), "/", len(xlsx))

## Linkage check against Corral

Classify every XLSX row as `matched`, `orphan`, or `ambiguous`. Ambiguous = the parsed ID points to a `TdrTransactionID` that doesn't exist in `dbo.TdrTransaction` (data-entry error in the XLSX).

In [ ]:
engine = get_engine()
with engine.connect() as conn:
    valid_ids = pd.read_sql(
        text("SELECT TdrTransactionID FROM dbo.TdrTransaction"),
        conn,
    )
valid_id_set = set(valid_ids["TdrTransactionID"].astype(int))
print(f"Corral TdrTransaction rows: {len(valid_id_set):,}")

def classify(tid) -> str:
    if tid is None or pd.isna(tid):
        return "orphan"
    return "matched" if int(tid) in valid_id_set else "ambiguous"

xlsx["LinkageStatus"] = xlsx["ParsedTdrTransactionID"].map(classify)
xlsx["LinkageStatus"].value_counts()

## Projection + normalization

Select only the XLSX-unique columns (category 3 from `erd/xlsx_decomposition.md`) plus keys and bookkeeping. Drop `Status Jan 2026`.

In [ ]:
def to_date(x) -> object:
    if x is None or (isinstance(x, float) and pd.isna(x)) or str(x).strip() == "":
        return None
    try:
        return pd.to_datetime(x, errors="coerce").date()
    except Exception:
        return None

def to_int(x) -> object:
    if x is None or (isinstance(x, float) and pd.isna(x)) or str(x).strip() == "":
        return None
    try:
        return int(float(str(x).strip()))
    except (ValueError, TypeError):
        return None

def to_str(x) -> object:
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    s = str(x).strip()
    return s if s else None

tt = pd.DataFrame({
    "TdrTransactionID":            xlsx["ParsedTdrTransactionID"],
    "SyntheticTransactionID":      xlsx["TransactionID"].map(to_str),
    "AllocationNumber":            xlsx["Allocation Number"].map(to_str),
    "TransactionCreatedDate":      xlsx["Transaction Created Date"].map(to_date),
    "TransactionAcknowledgedDate": xlsx["Transaction Acknowledged Date"].map(to_date),
    "DevelopmentType":             xlsx["Development Type"].map(to_str),
    "DetailedDevelopmentType":     xlsx["Detailed Development Type"].map(to_str),
    "TrpaMouProjectNumber":        xlsx["TRPA/MOU Project #"].map(to_str),
    "YearBuilt":                   xlsx["Year Built"].map(to_int),
    "PmYearBuilt":                 xlsx["PM Year Built"].map(to_int),
    "SupplementalNotes":           xlsx["Notes"].map(to_str),
    "LinkageStatus":               xlsx["LinkageStatus"],
    "SourceFile":                  XLSX.name,
    "SourceRowNumber":             xlsx["SourceRowNumber"].astype(int),
    "LoadedAt":                    LOADED_AT,
})
print(f"Transition rows: {len(tt):,}")
tt.head(3)

## Per-column population rate

Sanity check. Bookkeeping columns should be 100%. Others reflect how sparse the analyst's XLSX is.

In [ ]:
pop = pd.DataFrame({
    "column": tt.columns,
    "non_null": [int(tt[c].notna().sum()) for c in tt.columns],
    "rate": [round(float(tt[c].notna().mean()), 4) for c in tt.columns],
})
pop

## Write

In [ ]:
csv_path = OUT / "corral_transition_table.csv"
tt.to_csv(csv_path, index=False)
print(f"Wrote {csv_path} ({len(tt):,} rows)")

try:
    parquet_path = OUT / "corral_transition_table.parquet"
    tt.to_parquet(parquet_path, index=False)
    print(f"Wrote {parquet_path}")
except Exception as e:  # pyarrow optional in arcgispro-py3
    print(f"Skipped parquet write: {e}")

In [ ]:
summary = {
    "source_file": XLSX.name,
    "loaded_at": LOADED_AT,
    "rows": int(len(tt)),
    "linkage_status_counts": tt["LinkageStatus"].value_counts().to_dict(),
    "population": pop.to_dict("records"),
}
(OUT / "transition_table_summary.json").write_text(
    json.dumps(summary, indent=2, default=str), encoding="utf-8"
)
print(f"Wrote {OUT / 'transition_table_summary.json'}")
summary["linkage_status_counts"]